# 04. Model comparison and additional features

We compare OLS, Lasso, Ridge, Random Forest and Gradient Boosting using the same 13 features. For each model with hyperparameters, we select the candidate with the lowest mean MAE across five expanding-window splits before 2022. We then refit on all data through 2021 and score the same 365 days of 2022. The previous day's observed visits are available for each new forecast. The year 2023 is reserved for final evaluation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from features import build_features

project_dir = Path.cwd()
if project_dir.name == "analysis":
    project_dir = project_dir.parent

validation_start = pd.Timestamp("2022-01-01")
final_test_start = pd.Timestamp("2023-01-01")
visits = pd.read_parquet(
    project_dir / "data/processed/df_cleaned.parquet",
    filters=[("date", "<", final_test_start)],
)["visits"].sort_index()
assert visits.index.equals(pd.date_range("2017-01-01", "2022-12-31"))

X, y = build_features(visits)
assert X.shape[1] == 13 and X.index[0] == pd.Timestamp("2017-01-08")

train = X.index < validation_start
X_train, X_validation = X.loc[train], X.loc[~train]
y_train, y_validation = y.loc[train], y.loc[~train]
assert X_validation.index.equals(pd.date_range("2022-01-01", "2022-12-31"))

folds = list(TimeSeriesSplit(n_splits=5).split(X_train))
display(pd.DataFrame([
    {"Train through": X_train.index[tr[-1]].date(),
     "Validation start": X_train.index[va[0]].date(),
     "Validation end": X_train.index[va[-1]].date()}
    for tr, va in folds
]))

,Train through,Validation start,Validation end
0,2017-11-07,2017-11-08,2018-09-06
1,2018-09-06,2018-09-07,2019-07-06
2,2019-07-06,2019-07-07,2020-05-04
3,2020-05-04,2020-05-05,2021-03-03
4,2021-03-03,2021-03-04,2021-12-31


## 1. Ordinary least squares

OLS gives us a reference with the exact features and dates from notebook 03. We record its mean MAE across the earlier folds, as well as its MAE and R² on 2022.

In [2]:
from sklearn.model_selection import cross_val_score

results = []

def record_result(name, model, cv_mae, parameters):
    predictions = model.predict(X_validation)
    results.append({
        "Model": name,
        "Best parameters": parameters,
        "Earlier CV MAE": cv_mae,
        "2022 MAE": mean_absolute_error(y_validation, predictions),
        "2022 R²": r2_score(y_validation, predictions),
    })
    display(pd.DataFrame([results[-1]]).round(4))

ols = LinearRegression()
ols_cv_mae = -cross_val_score(
    ols, X_train, y_train, cv=folds, scoring="neg_mean_absolute_error"
).mean()
ols.fit(X_train, y_train)
record_result("OLS", ols, ols_cv_mae, {})
assert round(results[-1]["2022 MAE"], 2) == 52.96

,Model,Best parameters,Earlier CV MAE,2022 MAE,2022 R²
0,OLS,{},69.9138,52.9581,0.8027


## 2. Lasso and Ridge

We scale the inputs inside each training fold so the penalty is comparable across columns. Lasso can set coefficients to zero, while Ridge shrinks them. We include small penalties to see whether either model benefits from staying close to OLS.

In [3]:
def tune_and_record(name, estimator, grid):
    search = GridSearchCV(
        estimator, grid, cv=folds, scoring="neg_mean_absolute_error",
        n_jobs=2, error_score="raise",
    )
    search.fit(X_train, y_train)
    record_result(name, search.best_estimator_, -search.best_score_, search.best_params_)
    return search

alphas = np.logspace(-4, 3, 15)
lasso = tune_and_record(
    "Lasso", make_pipeline(StandardScaler(), Lasso(max_iter=20_000)),
    {"lasso__alpha": alphas},
)
ridge = tune_and_record(
    "Ridge", make_pipeline(StandardScaler(), Ridge()),
    {"ridge__alpha": alphas},
)
lasso_coefficients = lasso.best_estimator_.named_steps["lasso"].coef_
print(f"Lasso selected {(lasso_coefficients != 0).sum()} of {X.shape[1]} features.")

,Model,Best parameters,Earlier CV MAE,2022 MAE,2022 R²
0,Lasso,{'lasso__alpha': 1.0},69.8204,53.3139,0.7995


,Model,Best parameters,Earlier CV MAE,2022 MAE,2022 R²
0,Ridge,{'ridge__alpha': 3.1622776601683795},69.8563,52.9863,0.8024


Lasso selected 13 of 13 features.


## 3. Ensemble methods

Random Forest averages many trees fitted on resampled data. Gradient Boosting adds trees sequentially to correct earlier errors. The grids compare tree size and regularization at a manageable cost. The selected settings are the best among the candidates tested on the earlier folds.

In [4]:
forest = tune_and_record(
    "Random Forest",
    RandomForestRegressor(n_estimators=120, random_state=42),
    {"max_depth": [4, 8, None], "min_samples_leaf": [2, 8, 20],
     "max_features": [0.5, 1.0]},
)
boosting = tune_and_record(
    "Gradient Boosting",
    GradientBoostingRegressor(random_state=42),
    {"n_estimators": [100, 250], "learning_rate": [0.03, 0.1],
     "max_depth": [1, 2, 3], "min_samples_leaf": [5, 20]},
)

,Model,Best parameters,Earlier CV MAE,2022 MAE,2022 R²
0,Random Forest,"{'max_depth': None, 'max_features': 0.5, 'min_...",101.6649,63.8923,0.7012


,Model,Best parameters,Earlier CV MAE,2022 MAE,2022 R²
0,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 2, 'min_sa...",96.4289,60.2756,0.751


## 4. Results

The table sorts the models by 2022 MAE, measured in visits per day. The earlier CV MAE is the score used to choose each model's hyperparameters. Use both columns to assess whether the ranking carries across periods.

We also test one school-holiday indicator for Lyon (Zone A), using full days from the historical education ministry calendar and the official bulletin supplement for 2017. The return-to-school date is excluded. Calendar intervals and source links are stored in [the input CSV](../data/calendars/lyon-school-holidays-2017-2022.csv). We first test OLS. If its 2022 MAE falls, we compare the other models with their existing hyperparameters to isolate the effect of the new input. These are development comparisons.

In [5]:
summary = pd.DataFrame(results).sort_values("2022 MAE").reset_index(drop=True)
summary["2022 MAE gain vs OLS"] = results[0]["2022 MAE"] - summary["2022 MAE"]
display(summary.round(4))

from sklearn.base import clone

periods = pd.read_csv(
    project_dir / "data/calendars/lyon-school-holidays-2017-2022.csv",
    sep=";", parse_dates=["debut_analyse_inclus", "fin_analyse_exclue"],
)
school_holiday = np.zeros(len(X), dtype=bool)
for start, end in periods[["debut_analyse_inclus", "fin_analyse_exclue"]].itertuples(index=False, name=None):
    school_holiday |= (X.index >= start) & (X.index < end)
assert school_holiday.any() and not school_holiday.all()
X_school = X.assign(school_holiday=school_holiday.astype(int))
X_school_train = X_school.loc[train]
X_school_validation = X_school.loc[~train]

def compare_school_holidays(name, estimator):
    baseline = summary.loc[summary["Model"].eq(name)].iloc[0]
    model = clone(estimator)
    earlier_mae = -cross_val_score(
        model, X_school_train, y_train, cv=folds, scoring="neg_mean_absolute_error"
    ).mean()
    model.fit(X_school_train, y_train)
    predictions = model.predict(X_school_validation)
    validation_mae = mean_absolute_error(y_validation, predictions)
    return {
        "Model": name,
        "Earlier CV MAE without": baseline["Earlier CV MAE"],
        "Earlier CV MAE with": earlier_mae,
        "2022 MAE without": baseline["2022 MAE"],
        "2022 MAE with": validation_mae,
        "2022 MAE gain": baseline["2022 MAE"] - validation_mae,
    }

school_results = [compare_school_holidays("OLS", ols)]
if school_results[0]["2022 MAE gain"] > 0:
    for name, search in [
        ("Ridge", ridge), ("Lasso", lasso),
        ("Gradient Boosting", boosting), ("Random Forest", forest),
    ]:
        school_results.append(compare_school_holidays(name, search.best_estimator_))

school_summary = pd.DataFrame(school_results)
display(school_summary.round(4))

,Model,Best parameters,Earlier CV MAE,2022 MAE,2022 R²,2022 MAE gain vs OLS
0,OLS,{},69.9138,52.9581,0.8027,0.0000
1,Ridge,{'ridge__alpha': 3.1622776601683795},69.8563,52.9863,0.8024,-0.0282
2,Lasso,{'lasso__alpha': 1.0},69.8204,53.3139,0.7995,-0.3558
3,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 2, 'min_sa...",96.4289,60.2756,0.7510,-7.3175
4,Random Forest,"{'max_depth': None, 'max_features': 0.5, 'min_...",101.6649,63.8923,0.7012,-10.9342


,Model,Earlier CV MAE without,Earlier CV MAE with,2022 MAE without,2022 MAE with,2022 MAE gain
0,OLS,69.9138,70.3438,52.9581,52.0587,0.8995
1,Ridge,69.8563,70.2587,52.9863,52.0745,0.9117
2,Lasso,69.8204,70.1417,53.3139,52.4115,0.9025
3,Gradient Boosting,96.4289,95.0539,60.2756,59.5338,0.7418
4,Random Forest,101.6649,99.1374,63.8923,66.8278,-2.9355


**Feature decision.** Keep OLS with the original 13 features. Adding the school-holiday indicator reduced 2022 MAE from 52.96 to 52.06, while mean MAE across the five earlier validation periods rose from 69.91 to 70.34. The improvement did not carry across evaluation periods, so we drop the indicator from the selected model.

**Recent weather.** We compared OLS with and without maximum temperature and total precipitation from the Lyon-Bron area (45.72° N, 4.94° E, Europe/Paris). We tested windows of 1, 3 and 7 days ending the day before the target, using the mean of daily maxima and the precipitation sum. On the same 2022 dates, MAE was 52.91, 52.92 and 52.76, respectively, versus 52.96 for the reference. On the five earlier folds, mean MAE rose to 69.99, 69.96 and 70.44, respectively, versus 69.91. We excluded weather from the selected model. This was a retrospective test using ERA5 reanalysis, which is published with a delay. [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api).